In [ ]:
%pip install lightgbm scikit-learn pandas numpy matplotlib shap holidays openmeteo-requests requests-cache retry-requests entsoe-py yfinance pyarrow -q

# Alpine Arbitrage — Electricity Price Forecasting
### Team: alpine-arbitrage &nbsp;|&nbsp; Frigg S2S × EC EPFL × ACE Hackathon, May 2026

---
## 1. Methodology Overview

**SITUATION.** Europe's wholesale electricity markets are among the most complex and volatile in the world. Prices swing from negative values during sunny, windy weekends — when renewables flood the grid and gas plants must be paid to shut off — to €500+/MWh during energy crises, as witnessed in 2021–2022 when Russian gas supply collapsed. For energy infrastructure developers like Frigg, accurate price forecasting is the difference between a profitable project and a financial disaster.

**TASK.** Build a predictive system that forecasts Day-Ahead Auction (DAA) electricity prices in EUR/MWh for two European bidding zones — DE-LU (Germany-Luxembourg) and ES (Spain) — over *any* requested time horizon, from the next 24 hours to the next 20 years.

**ACTION.** We built a two-model system because the dominant drivers of price variance change fundamentally across horizons. For **short-term (0–7 days)** we use a quantile LightGBM model trained on 20 engineered features — selected from 41 candidates via SHAP + gain Borda rank fusion — calibrated with Mondrian Conformalized Quantile Regression (CQR) for coverage-guaranteed intervals. For **long-term (months to decades)** we built a structural merit-order model: stacking power plants by ascending marginal cost and finding where the stack meets demand, extended with forward fuel curves and a geopolitical risk detector, projected to 2045 via 200-draw Monte Carlo fan charts.

**RESULT.** Short-term model achieves **MAE of 6.01 EUR/MWh (DE-LU) and 5.15 EUR/MWh (ES)** on Q4 2025 holdout — approximately 5× better than the same-hour-last-week naive baseline. After Mondrian CQR calibration, prediction intervals achieve exactly 95% coverage. Long-term model achieves **€33/MWh MAE in normal market conditions** without using any historical prices as inputs. Main limitations: extreme events (negative prices, blackouts) are underestimated; ES shows a persistent underprediction from unmodelled MIBEL congestion premia.

---
## 2. Contents

| # | Section | What it covers |
|---|---------|----------------|
| 3 | Environment Setup | Dependencies, paths, data pipeline overview |
| 4 | Data & EDA | Price distributions, zone comparison, feature landscape |
| 5 | Feature Selection + Engineering | 41 → 20 features via SHAP + Borda fusion |
| 6 | Short-term Model | LightGBM training, validation metrics, MAE decomposition, SHAP |
| 7 | Uncertainty Calibration | Mondrian CQR — regime-specific 95% coverage intervals |
| 8 | Cross-zone Comparison | Where DE-LU and ES agree and diverge |
| 9 | Horizon Routing | Automatic ST↔LT switching logic + 30-day live demo |
| 10 | Long-term Model | Merit-order Monte Carlo (2026–2045), supply stacks, capacity roadmap |
| 11 | Evaluation Window | Competition submission — 24 slots, May 11 2026 |

---
## 3. Environment Setup

Two independent data pipelines feed two independent models:

| | Short-term (LightGBM) | Long-term (Merit-order MC) |
|-|----------------------|---------------------------|
| **Source** | `src/` | `longterm/src/` |
| **Processed** | `data/processed/final_dataset.parquet` | `longterm/data/processed/` |
| **Granularity** | Hourly, 2023–2026 | Monthly, 2018–2045 |
| **Target** | 24h DAA prices | Monthly avg prices + scenarios |

In [ ]:
import os, sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from IPython.display import Image, display

REPO   = Path('.').resolve()
SRC    = REPO / 'src'
LT_SRC = REPO / 'longterm' / 'src'
LT_ROOT = REPO / 'longterm'

for p in [str(SRC), str(LT_SRC)]:
    if p not in sys.path: sys.path.insert(0, p)

_env = REPO / '.env'
if _env.exists():
    for _line in _env.read_text().splitlines():
        if _line.strip() and not _line.startswith('#') and '=' in _line:
            k, v = _line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

ZONE_COLORS = {'DE-LU': '#1E88E5', 'ES': '#FF5722'}
ZONES = ['DE-LU', 'ES']
print('Repo:', REPO)
print('ENTSOE_TOKEN:', 'set' if os.environ.get('ENTSOE_TOKEN') else 'MISSING')

---
## 4. Data & EDA

**Pipeline A** (short-term) ingests hourly ENTSOE prices, generation, crossborder flows, day-ahead forecasts, 26-station Open-Meteo weather, ECMWF ensemble spread, and yfinance fuel prices. Four stages: ingest → clean → align → feature engineering.

**Pipeline B** (long-term) ingests monthly ENTSOE installed capacity, annual generation, and fuel prices. Four stages produce marginal cost curves and a structural capacity roadmap to 2045.

> Pre-computed parquets already exist. Uncomment the pipeline calls to re-run from scratch.

In [ ]:
# ── Load short-term dataset ───────────────────────────────────────────────
# Uncomment to re-run Pipeline A:
# from pipeline import run; run()

DATA_PATH = REPO / 'data' / 'processed' / 'final_dataset.parquet'
df = pd.read_parquet(DATA_PATH)

print(f'Shape      : {df.shape}')
print(f'Zones      : {df.index.get_level_values("zone").unique().tolist()}')
print(f'Date range : {df.index.get_level_values("timestamp").min().date()} → {df.index.get_level_values("timestamp").max().date()}')
print(f'Rows/zone  : {df.groupby(level="zone").size().to_dict()}')

### 4a. Price distributions and time series

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for row, zone in enumerate(ZONES):
    zdf   = df.xs(zone, level='zone').sort_index()
    price = zdf['price'].dropna()

    ax = axes[row, 0]
    ax.plot(price.index, price.values, lw=0.4, color=ZONE_COLORS[zone], alpha=0.7)
    ax.axhline(price.mean(), color='black', lw=1, ls='--', alpha=0.5, label=f'Mean {price.mean():.0f}')
    ax.axhline(0, color='red', lw=0.6, ls=':', alpha=0.4)
    ax.set_title(f'{zone} — Hourly price (full history)', fontweight='bold')
    ax.set_ylabel('EUR/MWh'); ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    ax = axes[row, 1]
    ax.hist(price.clip(-100, 300), bins=80, color=ZONE_COLORS[zone], alpha=0.8, edgecolor='white', lw=0.3)
    ax.axvline(price.median(), color='black', lw=1.5, label=f'Median {price.median():.0f}')
    ax.axvline(0, color='red', lw=1, ls='--', alpha=0.6)
    ax.set_title(f'{zone} — Price distribution', fontweight='bold')
    ax.set_xlabel('EUR/MWh'); ax.set_ylabel('Count'); ax.legend(fontsize=8)

    ax = axes[row, 2]
    hourly = price.groupby(price.index.hour).mean()
    ax.bar(hourly.index, hourly.values, color=ZONE_COLORS[zone], alpha=0.85)
    ax.set_title(f'{zone} — Average price by hour (UTC)', fontweight='bold')
    ax.set_xlabel('Hour'); ax.set_ylabel('Mean EUR/MWh'); ax.set_xticks(range(0, 24, 2))

plt.suptitle('Price overview — DE-LU vs ES (2023–2026)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_price_overview.png', dpi=120, bbox_inches='tight')
plt.show()

### 4b. Side-by-side zone comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
for zone in ZONES:
    zdf = df.xs(zone, level='zone').sort_index()
    monthly = zdf['price'].resample('ME').mean()
    ax.plot(monthly.index, monthly.values, lw=2, label=zone, color=ZONE_COLORS[zone])
ax.set_title('Monthly average price', fontweight='bold')
ax.set_ylabel('EUR/MWh'); ax.legend(); ax.axhline(0, color='grey', lw=0.5, ls=':')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

ax = axes[0, 1]
data_by_month = {zone: [df.xs(zone, level='zone')['price'].dropna()
    [df.xs(zone, level='zone')['price'].dropna().index.month == m].values
    for m in range(1, 13)] for zone in ZONES}
positions_de = np.arange(1, 13) - 0.2
positions_es = np.arange(1, 13) + 0.2
bp1 = ax.boxplot(data_by_month['DE-LU'], positions=positions_de, widths=0.35,
    patch_artist=True, boxprops=dict(facecolor='#90CAF9', alpha=0.7),
    medianprops=dict(color='#1E88E5', lw=2), showfliers=False)
bp2 = ax.boxplot(data_by_month['ES'], positions=positions_es, widths=0.35,
    patch_artist=True, boxprops=dict(facecolor='#FFAB91', alpha=0.7),
    medianprops=dict(color='#FF5722', lw=2), showfliers=False)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=8)
ax.set_title('Price distribution by month', fontweight='bold'); ax.set_ylabel('EUR/MWh')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['DE-LU', 'ES'], fontsize=9)

ax = axes[1, 0]
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
x = np.arange(7); width = 0.35
for i, zone in enumerate(ZONES):
    zdf = df.xs(zone, level='zone').sort_index()
    wd  = zdf['price'].dropna().groupby(zdf['price'].dropna().index.dayofweek).mean()
    ax.bar(x + (i-0.5)*width, wd.values, width, label=zone, color=ZONE_COLORS[zone], alpha=0.85)
ax.set_title('Average price by weekday', fontweight='bold')
ax.set_ylabel('EUR/MWh'); ax.set_xticks(x); ax.set_xticklabels(days); ax.legend()

ax = axes[1, 1]
de_price = df.xs('DE-LU', level='zone')['price'].dropna()
es_price = df.xs('ES', level='zone')['price'].dropna()
common   = de_price.index.intersection(es_price.index)
corr     = np.corrcoef(de_price[common], es_price[common])[0, 1]
ax.scatter(de_price[common], es_price[common], s=1, alpha=0.15, color='#7B1FA2')
ax.set_xlabel('DE-LU (EUR/MWh)'); ax.set_ylabel('ES (EUR/MWh)')
ax.set_title(f'Price correlation (r={corr:.2f})', fontweight='bold')
m = min(de_price[common].min(), es_price[common].min())
M = max(de_price[common].quantile(0.99), es_price[common].quantile(0.99))
ax.plot([m,M],[m,M],'k--',lw=0.8,alpha=0.4); ax.set_xlim(m,M); ax.set_ylim(m,M)

plt.suptitle('DE-LU vs ES — Side-by-side market comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_zone_comparison_eda.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Feature Selection + Engineering

Started with 41 candidate features. Applied SHAP + gain Borda rank fusion on the 2025 holdout independently per zone. Took the union of top-15 per zone → 20-feature shared schema. Each zone gets its best 15; the 5 zone-specific extras are NaN for the other zone — LightGBM skips NaN-only columns at split time.

**Dropped (low Borda in both zones):** multi-city weather aggregates, calendar circulars, regime flags, most neighbor spreads, ensemble uncertainty, gas and coal prices.

| Group | Features | Both zones? |
|-------|----------|------------|
| Price history | `lag_1`, `lag_24`, `lag_168`, `price_roll_24h`, `price_roll_168h`, `price_roll_std_168h` | ✓ |
| Grid state | `residual_load_ramp_forecast`, `residual_load_ramp`, `residual_load_forecast`, `residual_load`, `renewable_penetration_forecast` | ✓ |
| Generation | `hydro_generation`, `wind_generation`*, `solar_generation`* | *zone-specific |
| Cross-border | `net_imports`, `cross_zone_lag24`, `NL_price_lag24`*, `CH_price_lag24`* | *DE-LU only |
| Fuel / carbon | `carbon_price` | ✓ |
| Load | `load`* | *DE-LU only |

In [ ]:
# Feature availability heatmap — shows which feature groups are present per zone
FEATURE_GROUPS = {
    'Price lags':      ['lag_1','lag_24','lag_168','price_roll_24h','price_roll_168h','price_roll_std_168h'],
    'Supply/demand':   ['load','wind_generation','solar_generation','hydro_generation','nuclear_generation'],
    'Multi-city wx':   ['wind_speed_agg','wind_speed_cubed','solar_radiation_agg','solar_hour_interaction','temperature_agg','temperature_sq'],
    'DE-LU cross-bdr': ['DK_wind_speed','DK_wind_speed_cubed','CH_precipitation','CH_precip_7d_sum'],
    'ES hydro/nuclear':['ES_hydro_precipitation','ES_hydro_precip_7d_sum','nuclear_available_mw'],
    'Fuel/carbon':     ['gas_price','carbon_price','coal_price'],
    'Derived gen':     ['residual_load','renewable_penetration','residual_load_ramp'],
    'Neighbor prices': ['FR_price_lag24','NL_price_lag24','CH_price_lag24','DK_price_lag24'],
    'Spreads':         ['DE_LU_FR_spread','DE_LU_NL_spread','DE_LU_CH_spread','ES_FR_spread'],
    'Calendar':        ['hour_sin','weekday_sin','month_sin','is_holiday','days_to_holiday'],
    'Regime flags':    ['crisis_period','is_peak','negative_price_lag24'],
    'Ensemble uncert': ['wind_ensemble_std','solar_ensemble_std'],
}

fig, ax = plt.subplots(figsize=(14, 5))
groups  = list(FEATURE_GROUPS.keys())
matrix  = np.zeros((len(ZONES), len(groups)))
for j, (grp, feats) in enumerate(FEATURE_GROUPS.items()):
    for i, zone in enumerate(ZONES):
        zdf = df.xs(zone, level='zone')
        present = [f for f in feats if f in zdf.columns and zdf[f].notna().mean() > 0.1]
        matrix[i, j] = len(present) / max(len(feats), 1)

im = ax.imshow(matrix, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(groups)))
ax.set_xticklabels(groups, rotation=35, ha='right', fontsize=9)
ax.set_yticks([0,1]); ax.set_yticklabels(['DE-LU','ES'], fontsize=11)
for i in range(len(ZONES)):
    for j in range(len(groups)):
        pct = matrix[i,j]
        ax.text(j, i, f'{pct:.0%}', ha='center', va='center',
                fontsize=9, color='white' if pct > 0.6 else 'black', fontweight='bold')
plt.colorbar(im, ax=ax, label='Coverage (% non-NaN)', shrink=0.6)
ax.set_title('All 41 candidate features — coverage by zone (shaded = selected)', fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_feature_coverage.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Short-term Model — LightGBM Training + Validation

Three quantile models per zone (q=0.025, 0.45, 0.975), 3000 boosting rounds with early stopping on Q4 2025 validation set. Training window: **2023-05-01 → 2025-09-30**. Validation: **Q4 2025** (Oct–Dec). Calibration: **Jan–May 2026** (CQR, next section).

Scoring target is pinball loss at **q=0.45** — overestimation penalised 1.22× more.

In [ ]:
import lightgbm as lgb
import holidays as hdays
from sklearn.metrics import mean_pinball_loss

FEATURES = [f for f in [
    'lag_1','lag_24','lag_168',
    'price_roll_24h','price_roll_168h','price_roll_std_168h',
    'residual_load_ramp_forecast','residual_load_ramp',
    'residual_load_forecast','residual_load',
    'renewable_penetration_forecast',
    'hydro_generation','wind_generation','solar_generation',
    'net_imports','cross_zone_lag24',
    'NL_price_lag24','CH_price_lag24',
    'load',
    'carbon_price',
] if f in df.columns]

TARGET    = 'price'
QUANTILES = [0.025, 0.45, 0.975]
TRAIN_END = '2025-10-01'
VAL_END   = '2026-01-01'
CAL_END   = '2026-05-10'

LGB_BASE = dict(objective='quantile', metric='quantile', n_estimators=3000,
    learning_rate=0.05, num_leaves=127, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, verbose=-1)

def pinball(y_true, y_pred, q):
    return float(mean_pinball_loss(y_true, y_pred, alpha=q))

def coverage(y_true, lo, hi):
    return float(((y_true >= lo) & (y_true <= hi)).mean())

print(f'Active features: {len(FEATURES)}')
print('Training...')
all_models, all_val = {}, {}
for zone in ZONES:
    zdf   = df.xs(zone, level='zone').sort_index()
    train = zdf[zdf.index < TRAIN_END].dropna(subset=[TARGET])
    val   = zdf[(zdf.index >= TRAIN_END) & (zdf.index < VAL_END)].dropna(subset=[TARGET])
    X_tr, y_tr = train[FEATURES], train[TARGET]
    X_va, y_va = val[FEATURES],   val[TARGET]
    print(f'  {zone}: train={len(X_tr):,}  val={len(X_va):,}')
    qm, vp = {}, {}
    for q in QUANTILES:
        m = lgb.LGBMRegressor(**{**LGB_BASE, 'alpha': q})
        fit_kw = {'eval_set': [(X_va, y_va)],
                  'callbacks': [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]}
        m.fit(X_tr, y_tr, **fit_kw) if len(X_va) > 0 else m.fit(X_tr, y_tr)
        vp[q] = m.predict(X_va) if len(X_va) > 0 else np.array([])
        qm[q] = m
        if len(X_va): print(f'    q={q:.3f}  iter={m.best_iteration_}  pb={pinball(y_va.values, vp[q], q):.4f}')
    all_models[zone] = qm
    all_val[zone]    = {'preds': vp, 'actual': y_va, 'X': X_va}
print('Done.')

### 6a. Validation metrics

In [ ]:
print('=' * 60)
print('VALIDATION RESULTS  (Q4 2025 holdout)')
print('=' * 60)
val_summary = {}
for zone in ZONES:
    y     = all_val[zone]['actual'].values
    p025  = all_val[zone]['preds'][0.025]
    p50   = all_val[zone]['preds'][0.45]
    p975  = all_val[zone]['preds'][0.975]
    naive = all_val[zone]['X']['lag_168'].values
    mae   = np.abs(y - p50).mean()
    val_summary[zone] = {'MAE': mae, 'Pinball': pinball(y, p50, 0.45),
                          'Coverage': coverage(y, p025, p975)*100}
    print(f'\n  {zone}')
    print(f'  MAE (p50)        : {mae:.2f} EUR/MWh')
    print(f'  Pinball q=0.45   : {pinball(y, p50, 0.45):.4f}')
    print(f'  Coverage p025-975: {coverage(y, p025, p975)*100:.1f}%')
    print(f'  Band width       : {(p975-p025).mean():.2f} EUR/MWh')
    print(f'  Naive MAE        : {np.abs(y-naive).mean():.2f} EUR/MWh')
    print(f'  vs Naive         : {mae / np.abs(y-naive).mean():.2f}× better')

### 6b. MAE over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for ax, zone in zip(axes, ZONES):
    y_s   = all_val[zone]['actual']
    p50_s = pd.Series(all_val[zone]['preds'][0.45], index=y_s.index)
    daily = (y_s - p50_s).abs().groupby(y_s.index.date).mean()
    daily.index = pd.to_datetime(daily.index)
    roll7 = daily.rolling(7, center=True, min_periods=3).mean()
    ax.bar(daily.index, daily.values, color=ZONE_COLORS[zone], alpha=0.4, width=0.9)
    ax.plot(daily.index, roll7.values, color=ZONE_COLORS[zone], lw=2, label='7-day avg')
    ax.axhline(daily.mean(), color='black', lw=1, ls='--', alpha=0.6, label=f'Mean {daily.mean():.1f}')
    worst = daily.idxmax()
    ax.annotate(f'{worst.strftime("%b %d")}\n{daily[worst]:.0f}', xy=(worst, daily[worst]),
               xytext=(0,12), textcoords='offset points', ha='center', fontsize=8, color='red',
               arrowprops=dict(arrowstyle='->', color='red', lw=0.8))
    ax.set_ylabel('Daily MAE (EUR/MWh)'); ax.set_title(f'{zone}', fontweight='bold'); ax.legend(fontsize=9)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.suptitle('MAE over time — Q4 2025 holdout', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_mae_overtime.png', dpi=120, bbox_inches='tight')
plt.show()

### 6c. MAE decomposition — hour, weekday, month

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for row, zone in enumerate(ZONES):
    y_s   = all_val[zone]['actual']
    p50_s = pd.Series(all_val[zone]['preds'][0.45], index=y_s.index)
    err   = (y_s - p50_s).abs()

    ax = axes[row, 0]
    by_hour = err.groupby(err.index.hour).mean()
    ax.bar(by_hour.index, by_hour.values, color=ZONE_COLORS[zone], alpha=0.85)
    ax.set_title(f'{zone} — MAE by hour (UTC)', fontweight='bold')
    ax.set_xlabel('Hour'); ax.set_ylabel('MAE (EUR/MWh)'); ax.set_xticks(range(0,24,2))

    ax = axes[row, 1]
    by_wd = err.groupby(err.index.dayofweek).mean()
    days  = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    bars  = ax.bar(days, by_wd.values, color=ZONE_COLORS[zone], alpha=0.85)
    for i in [5,6]: bars[i].set_alpha(0.5)
    ax.set_title(f'{zone} — MAE by weekday', fontweight='bold'); ax.set_ylabel('MAE (EUR/MWh)')

    ax = axes[row, 2]
    by_mo = err.groupby(err.index.month).mean()
    _all_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    months = [_all_months[m-1] for m in by_mo.index]
    cvals  = by_mo.values
    cols   = plt.cm.RdYlGn_r(plt.Normalize(cvals.min(), cvals.max())(cvals))
    ax.bar(months, cvals, color=cols, alpha=0.9)
    ax.set_title(f'{zone} — MAE by month', fontweight='bold'); ax.set_ylabel('MAE (EUR/MWh)')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('MAE decomposition — where and when the model struggles', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_mae_decomposition.png', dpi=120, bbox_inches='tight')
plt.show()

### 6d. SHAP — what drives the model

In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable,'-m','pip','install','--break-system-packages','shap'], check=True)
    import shap

rng = np.random.default_rng(42)
shap_data = {}
for zone in ZONES:
    X_val    = all_val[zone]['X']
    n        = min(800, len(X_val))
    idx      = rng.choice(len(X_val), size=n, replace=False)
    X_samp   = X_val.iloc[idx]
    explainer = shap.TreeExplainer(all_models[zone][0.45])
    sv        = explainer.shap_values(X_samp)
    shap_data[zone] = {'sv': sv, 'X': X_samp, 'base': float(explainer.expected_value)}
    print(f'{zone}: SHAP computed (n={n}, base={explainer.expected_value:.1f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
GROUP_C = {
    'lag_':'#E53935','price_roll':'#E53935','wind':'#1E88E5','solar':'#FDD835',
    'load':'#8E24AA','residual':'#AB47BC','renewable':'#AB47BC','net_imports':'#AB47BC',
    'gas_':'#6D4C41','carbon':'#6D4C41','coal':'#6D4C41',
    'FR_':'#00ACC1','NL_':'#00ACC1','CH_':'#00ACC1','DK_':'#00ACC1',
    'cross_zone':'#00ACC1','hydro':'#78909C','nuclear':'#78909C',
}
def gcol(f):
    for k,c in GROUP_C.items():
        if k in f: return c
    return '#BDBDBD'

for ax, zone in zip(axes, ZONES):
    sv       = shap_data[zone]['sv']
    mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=FEATURES).sort_values()
    signed   = pd.Series(sv.mean(axis=0), index=FEATURES).loc[mean_abs.index]
    colors   = [gcol(f) for f in mean_abs.index]
    ax.barh(range(len(mean_abs)), mean_abs.values, color=colors, alpha=0.8)
    for i, (feat, val) in enumerate(signed.items()):
        sym = '▶' if val > 0 else '◀'
        col = '#C62828' if val > 0 else '#1565C0'
        ax.text(mean_abs[feat] + mean_abs.max()*0.01, i, sym, va='center', fontsize=7, color=col)
    ax.set_yticks(range(len(mean_abs)))
    ax.set_yticklabels(mean_abs.index, fontsize=7)
    ax.set_xlabel('Mean |SHAP| (EUR/MWh)')
    ax.set_title(f'{zone} — SHAP (▶ raises price  ◀ lowers)', fontweight='bold', fontsize=10)

plt.suptitle('SHAP Feature Importance — p50 model, Q4 2025 validation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_shap_global.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Uncertainty Calibration — Mondrian CQR

Raw quantile models are well-calibrated in point forecast but their intervals (`p025`–`p975`) don't reliably achieve 95% coverage. **Mondrian CQR** fixes this by computing a correction `Q_hat` from a held-out calibration set (Jan–May 2026), split into two regime buckets:

- **Bucket 0** — normal weekday: tighter intervals
- **Bucket 1** — weekend / holiday / bridge day: wider intervals (more uncertainty)

All 24 May 11 eval slots are a Sunday → bucket 1.

In [ ]:
def _mondrian_bucket(ts, cal):
    d = ts.date()
    if ts.dayofweek >= 5: return 1
    if d in cal: return 1
    if (ts - pd.Timedelta(days=1)).date() in cal: return 1
    if (ts + pd.Timedelta(days=1)).date() in cal: return 1
    return 0

_COUNTRY = {'DE-LU': 'DE', 'ES': 'ES'}
cqr = {}
for zone in ZONES:
    zdf    = df.xs(zone, level='zone').sort_index()
    cal_df = zdf[(zdf.index >= VAL_END) & (zdf.index < CAL_END)].dropna(subset=[TARGET])
    X_cal  = cal_df[FEATURES]; y_cal = cal_df[TARGET].values; n = len(y_cal)
    p025 = all_models[zone][0.025].predict(X_cal)
    p50  = all_models[zone][0.45].predict(X_cal)
    p975 = all_models[zone][0.975].predict(X_cal)
    p025 = np.minimum(p025, p50); p975 = np.maximum(p975, p50)
    scores   = np.maximum(p025 - y_cal, y_cal - p975)
    q_hat_iv = float(np.quantile(scores, min(0.95*(1+1/n), 1.0)))
    q_hat_50 = float(np.quantile(y_cal - p50, min(0.45*(1+1/n), 1.0)))
    hol_cal  = hdays.country_holidays(_COUNTRY[zone],
                 years=range(int(cal_df.index.year.min()), int(cal_df.index.year.max())+2))
    buckets  = np.array([_mondrian_bucket(ts, hol_cal) for ts in cal_df.index])
    mondrian = {}
    for b in [0, 1]:
        mask_b = buckets == b; n_b = int(mask_b.sum())
        mondrian[b] = float(np.quantile(scores[mask_b], min(0.95*(1+1/n_b),1.0))) if n_b>=50 else q_hat_iv
        cov_b = float(((y_cal[mask_b]>=p025[mask_b]-mondrian[b]) &
                       (y_cal[mask_b]<=p975[mask_b]+mondrian[b])).mean())*100 if n_b>0 else 0
        print(f'  {zone} bucket {b} (n={n_b:,}): Q_hat={mondrian[b]:.2f}  coverage={cov_b:.1f}%')
    cov_raw = float(((y_cal>=p025)&(y_cal<=p975)).mean())*100
    cov_cal = float(((y_cal>=p025-q_hat_iv)&(y_cal<=p975+q_hat_iv)).mean())*100
    print(f'  {zone}: raw cov={cov_raw:.1f}% → calibrated={cov_cal:.1f}%  p50 shift={q_hat_50:+.2f}')
    cqr[zone] = {'interval': q_hat_iv, 'p50': q_hat_50, 'mondrian': mondrian}
    print()

---
## 8. Cross-zone Comparison — DE-LU vs ES

| Dimension | DE-LU | ES |
|-----------|-------|----|
| Dominant renewable | Wind (north Germany) | Solar (Andalusia, Murcia) |
| Nuclear | Zero (phase-out Apr 2023) | ~7 GW active |
| Interconnection | 8 neighbours, tightly coupled | 1 neighbour (France), partially isolated |
| Hydro role | Minor | Major swing supply source |
| Negative prices | Frequent (wind surplus weekends) | Less common |

In [ ]:
rows = []
for zone in ZONES:
    y    = all_val[zone]['actual'].values
    p50  = all_val[zone]['preds'][0.45]
    p025 = all_val[zone]['preds'][0.025]
    p975 = all_val[zone]['preds'][0.975]
    naive = all_val[zone]['X']['lag_168'].values
    rows.append({'Zone': zone,
                 'MAE': round(np.abs(y-p50).mean(),2),
                 'Pinball q=0.45': round(pinball(y, p50, 0.45),4),
                 'Coverage': f"{coverage(y,p025,p975)*100:.1f}%",
                 'Band width': round((p975-p025).mean(),2),
                 'vs Naive': f"{np.abs(y-naive).mean()/np.abs(y-p50).mean():.1f}×"})
pd.DataFrame(rows).set_index('Zone')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
for zone in ZONES:
    y_s   = all_val[zone]['actual']
    p50_s = pd.Series(all_val[zone]['preds'][0.45], index=y_s.index)
    daily = (y_s - p50_s).abs().groupby(y_s.index.date).mean()
    daily.index = pd.to_datetime(daily.index)
    roll7 = daily.rolling(7, center=True, min_periods=3).mean()
    ax.plot(daily.index, roll7.values, lw=2, label=f'{zone} (7-day avg)', color=ZONE_COLORS[zone])
    ax.fill_between(daily.index, 0, daily.values, alpha=0.12, color=ZONE_COLORS[zone])
ax.set_ylabel('Daily MAE (EUR/MWh)'); ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.set_title('DE-LU vs ES — MAE over time (Q4 2025): where models agree and diverge', fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_crosszone_mae.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ranks = {}
for zone in ZONES:
    sv  = shap_data[zone]['sv']
    imp = pd.Series(np.abs(sv).mean(axis=0), index=FEATURES).sort_values(ascending=False)
    ranks[zone] = {feat: i+1 for i, feat in enumerate(imp.index)}

top15_de   = list(pd.Series(np.abs(shap_data['DE-LU']['sv']).mean(axis=0), index=FEATURES).sort_values(ascending=False).head(15).index)
top15_es   = list(pd.Series(np.abs(shap_data['ES']['sv']).mean(axis=0), index=FEATURES).sort_values(ascending=False).head(15).index)
union_feats = list(dict.fromkeys(top15_de + top15_es))[:20]

x = np.arange(len(union_feats)); w = 0.35
for i, zone in enumerate(ZONES):
    imp_zone = pd.Series(np.abs(shap_data[zone]['sv']).mean(axis=0), index=FEATURES)
    vals = [imp_zone.get(f, 0) for f in union_feats]
    ax.bar(x + (i-0.5)*w, vals, w, label=zone, color=ZONE_COLORS[zone], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(union_feats, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Mean |SHAP| (EUR/MWh)'); ax.legend(fontsize=10)
ax.set_title('Feature importance by zone — top 20 features (union)\nDifferences show which drivers dominate each market', fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_crosszone_shap.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'{"Feature":<35} {"DE-LU":>10} {"ES":>8} {"Shift":>10}')
print('-' * 68)
for feat in union_feats:
    r_de = ranks['DE-LU'].get(feat, '-'); r_es = ranks['ES'].get(feat, '-')
    shift = (r_de - r_es) if isinstance(r_de,int) and isinstance(r_es,int) else '-'
    arrow = 'DE-LU higher' if isinstance(shift,int) and shift>3 else 'ES higher' if isinstance(shift,int) and shift<-3 else 'similar'
    print(f'{feat:<35} #{r_de:>9} #{r_es:>7}   {arrow}')

---
## 9. Horizon Routing — ST ↔ LT Transition

Every prediction request is routed automatically based on how far the slot is from the training tail:

```
  slot_date ≤ training_tail + 7 days?
         │ YES                        │ NO
         ▼                            ▼
  LightGBM + Mondrian CQR    Seasonal median + trend
  Interval: Q_hat/bucket     Interval: resid_std × sqrt-scaled
```

**Why 7 days?** ENTSOE generation forecasts and Open-Meteo weather degrade past day 7. Beyond that, recursive lag-fill error compounds faster than the seasonal model's residual.

| Horizon | Model | Uncertainty |
|---------|-------|-------------|
| 0–7 days | Quantile LightGBM | Mondrian CQR Q_hat per bucket |
| 8 days – months | Seasonal median + trend | resid_std × 1.96 × sqrt(1 + excess/30) |
| Months – 2045 | Merit-order Monte Carlo | 200-draw fan chart |

In [ ]:
# Timeline diagram
fig, ax = plt.subplots(figsize=(14, 3))
tail   = pd.Timestamp('2026-05-07', tz='UTC')
cutoff = tail + pd.Timedelta(days=7)
eval_d = pd.Timestamp('2026-05-11', tz='UTC')
ax.barh(0, (cutoff-tail).days, left=0, height=0.4, color='#2196F3', label='LightGBM (≤7d)')
ax.barh(0, 60, left=(cutoff-tail).days, height=0.4, color='#FF9800', alpha=0.7, label='Seasonal median (>7d)')
ax.barh(0, 365*19, left=60, height=0.4, color='#9C27B0', alpha=0.4, label='Merit-order MC')
eval_offset = (eval_d - tail).days
ax.axvline(eval_offset, color='red', linewidth=2, linestyle='--')
ax.text(eval_offset+0.3, 0.25, 'Eval window\n(May 11)', color='red', fontsize=9)
ax.axvline(7, color='black', linewidth=1.5, linestyle=':')
ax.text(7.3, 0.25, '7d cutoff', fontsize=9)
ax.set_xlabel('Days from training tail (2026-05-07)')
ax.set_xlim(0, 100); ax.set_yticks([])
ax.legend(loc='upper right', fontsize=9)
ax.set_title('Horizon routing — which model handles which slots', fontweight='bold')
plt.tight_layout(); plt.show()

### 9b. Live 30-day Demo — colour-coded by regime

First 7 days → LightGBM (using hour×weekday historical feature profiles as proxy). Beyond 7 days → seasonal median + trend. Both regimes shown on the same chart.

In [ ]:
from model import build_longterm_model, predict_longterm_slot, SHORTTERM_DAYS

DEMO_START    = pd.Timestamp('2026-05-11 00:00', tz='UTC')
DEMO_DAYS     = 30
TRAINING_TAIL = pd.Timestamp(TRAIN_END, tz='UTC')

lt_models = {}
feature_profiles = {}
for zone in ZONES:
    zdf_zone = df.xs(zone, level='zone').sort_index()
    lt_models[zone] = build_longterm_model(zdf_zone, zone)
    recent = zdf_zone[zdf_zone.index >= '2025-01-01'].copy()
    recent['_h'] = recent.index.hour; recent['_d'] = recent.index.dayofweek
    feature_profiles[zone] = recent.groupby(['_h','_d'])[FEATURES].median()

demo_slots   = pd.date_range(DEMO_START, periods=DEMO_DAYS*24, freq='h', tz='UTC')
demo_results = {zone: [] for zone in ZONES}
for zone in ZONES:
    lt = lt_models[zone]; prof = feature_profiles[zone]
    for ts in demo_slots:
        hd = (ts - TRAINING_TAIL).total_seconds() / 86400.0
        if hd <= SHORTTERM_DAYS:
            h, d = ts.hour, ts.dayofweek
            row  = prof.loc[(h,d)].to_frame().T if (h,d) in prof.index else prof.iloc[[0]]
            p025 = float(all_models[zone][0.025].predict(row)[0])
            p50  = float(all_models[zone][0.45].predict(row)[0])
            p975 = float(all_models[zone][0.975].predict(row)[0])
            regime = 'ST'
        else:
            p025, p50, p975 = predict_longterm_slot(ts, lt_models[zone], hd)
            regime = 'LT'
        demo_results[zone].append({'ts': ts, 'p025': float(p025), 'p50': float(p50), 'p975': float(p975), 'regime': regime})
    demo_results[zone] = pd.DataFrame(demo_results[zone]).set_index('ts')
    r = demo_results[zone]
    print(f'{zone}: {(r["regime"]=="ST").sum()} ST (LightGBM)  |  {(r["regime"]=="LT").sum()} LT (seasonal)')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
for ax, zone in zip(axes, ZONES):
    r   = demo_results[zone].copy()
    for col in ['p025','p50','p975']: r[col] = pd.to_numeric(r[col], errors='coerce')
    r.index = r.index.tz_localize(None)
    st = r[r['regime']=='ST']; lt = r[r['regime']=='LT']
    st_ix = st.index; lt_ix = lt.index
    cutoff_ts = (TRAINING_TAIL + pd.Timedelta(days=SHORTTERM_DAYS)).tz_localize(None)
    if len(st):
        ax.fill_between(st_ix, st['p025'].astype(float), st['p975'].astype(float), alpha=0.15, color='#2196F3')
        ax.plot(st_ix, st['p50'].astype(float), color='#2196F3', lw=1.8, label=f'LightGBM ST ({len(st)} slots)')
    if len(lt):
        ax.fill_between(lt_ix, lt['p025'].astype(float), lt['p975'].astype(float), alpha=0.15, color='#FF9800')
        ax.plot(lt_ix, lt['p50'].astype(float), color='#FF9800', lw=1.8, label=f'Seasonal LT ({len(lt)} slots)')
    ax.axvline(cutoff_ts, color='black', lw=1.2, ls=':', label='7d cutoff')
    ax.set_ylabel('EUR/MWh'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_title(f'{zone} — 30-day forecast, colour by model regime', fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.suptitle('ST → LT routing: blue=LightGBM, orange=seasonal median', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 10. Long-term Model — Merit-order Monte Carlo (2026–2045)

The merit-order model stacks power plants by ascending marginal cost and finds where cumulative capacity meets demand. Forward fuel curves (Schwartz mean-reversion) replace spot prices; a geopolitical risk detector (Iacoviello GPR index) flags crisis regimes. 200-draw Monte Carlo fan charts capture structural uncertainty to 2045.

In [ ]:
# Load long-term datasets
# Uncomment to re-run Pipeline B:
# import importlib; import pipeline as lt_pipeline; importlib.reload(lt_pipeline); lt_pipeline.run()

mc_path  = LT_ROOT / 'data' / 'processed' / 'marginal_costs_monthly.parquet'
str_path = LT_ROOT / 'data' / 'processed' / 'structural_extended.parquet'
mc_df    = pd.read_parquet(mc_path)
str_df   = pd.read_parquet(str_path)
print('Marginal costs:', mc_df.shape, '| Structural:', str_df.shape)

In [ ]:
# Merit-order supply stacks — 2024
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
tech_colors = {
    'wind_solar':'#FDD835','hydro':'#29B6F6','nuclear':'#66BB6A','lignite':'#8D6E63',
    'coal':'#616161','gas_ccgt':'#FF7043','oil_peaker':'#E53935','biomass_other':'#A5D6A7'
}
for ax, zone in zip(axes, ZONES):
    zmc    = mc_df[mc_df['zone'] == zone]
    recent = zmc[zmc['month'].dt.year == 2024]
    by_tech = recent.groupby('tech')['mc_eur_per_mwh'].mean().sort_values()
    colors  = [tech_colors.get(t, '#BDBDBD') for t in by_tech.index]
    ax.barh(range(len(by_tech)), by_tech.values, color=colors, alpha=0.9)
    ax.set_yticks(range(len(by_tech))); ax.set_yticklabels(by_tech.index, fontsize=9)
    ax.set_xlabel('Marginal cost (EUR/MWh)')
    ax.set_title(f'{zone} — Supply stack (2024 avg)', fontweight='bold')
    for i, (t, v) in enumerate(by_tech.items()):
        ax.text(v+0.5, i, f'€{v:.0f}', va='center', fontsize=8)
plt.suptitle('Merit-order supply stacks — gas CCGT typically sets the marginal price', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_supply_stacks.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Hero long-term forecast (pre-generated)
hero_path = LT_ROOT / 'notebooks' / 'hero_long_term_forecast.png'
display(Image(filename=str(hero_path), width=1000))

In [ ]:
# Capacity roadmap to 2045
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cap_col_colors = {
    'capacity_wind_solar_gw':'#FDD835','capacity_hydro_gw':'#29B6F6',
    'capacity_nuclear_gw':'#66BB6A','capacity_lignite_gw':'#8D6E63',
    'capacity_coal_gw':'#616161','capacity_gas_ccgt_gw':'#FF7043',
    'capacity_oil_peaker_gw':'#E53935','capacity_biomass_other_gw':'#A5D6A7',
}
cap_cols = [c for c in str_df.columns if c.startswith('capacity_')]
for ax, zone in zip(axes, ZONES):
    try: zstr = str_df.xs(zone, level='zone')
    except: zstr = str_df
    years  = zstr.index.get_level_values('year') if 'year' in str_df.index.names else zstr.index
    bottom = np.zeros(len(years))
    for col in cap_cols:
        if col not in zstr.columns: continue
        vals  = zstr[col].values
        label = col.replace('capacity_','').replace('_gw','').replace('_',' ')
        ax.fill_between(years, bottom, bottom+vals,
            label=label, color=cap_col_colors.get(col,'#BDBDBD'), alpha=0.85)
        bottom += vals
    ax.set_title(f'{zone} — Capacity roadmap to 2045', fontweight='bold')
    ax.set_ylabel('GW'); ax.set_xlabel('Year')
    ax.legend(loc='upper left', fontsize=7, ncol=2)
    ax.axvline(2026, color='black', lw=1, ls='--', alpha=0.5)
plt.suptitle('Capacity roadmap: observed (2018–2025) + projected (2026–2045)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_capacity_roadmap.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 11. Evaluation Window — May 11, 2026 (24 slots)

Competition target: **2026-05-11 02:00 CEST → 2026-05-12 01:00 CEST** (00:00–23:00 UTC).  
Scoring: **Pinball loss q=0.45**. May 11 is a **Sunday** → Mondrian bucket 1 (wider intervals).  
All 24 slots routed through LightGBM — 4 days from training tail, well within 7-day ST regime.

In [ ]:
pred_path = REPO / 'alpine-arbitrage_predictions.csv'
preds = pd.read_csv(pred_path)
preds['timestamp'] = pd.to_datetime(preds['timestamp'])
print(f'Submission: {len(preds)} rows')
print(f'Range: {preds["timestamp"].iloc[0]} → {preds["timestamp"].iloc[-1]}')
preds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, zone in zip(axes, ZONES):
    ts   = preds['timestamp']
    p025 = preds[f'{zone} p025']
    p50  = preds[f'{zone} p50']
    p975 = preds[f'{zone} p975']
    ax.fill_between(ts, p025, p975, alpha=0.25, color=ZONE_COLORS[zone], label='95% CI (CQR)')
    ax.plot(ts, p50, lw=2.5, color=ZONE_COLORS[zone], marker='o', ms=4, label='p50 forecast')
    ax.axhline(p50.mean(), color='grey', lw=1, ls='--', alpha=0.6, label=f'Daily mean {p50.mean():.0f}')
    mid = p50.idxmin()
    ax.annotate(f'Solar trough\n{p50[mid]:.1f} EUR/MWh', xy=(ts[mid], p50[mid]),
               xytext=(0,-30), textcoords='offset points', ha='center', fontsize=8,
               color='darkorange', arrowprops=dict(arrowstyle='->', color='darkorange', lw=0.8))
    ax.set_title(f'{zone} — May 11, 2026 (24h eval window)', fontweight='bold')
    ax.set_ylabel('EUR/MWh'); ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
    ax.set_xlabel('CEST time')
plt.suptitle('alpine-arbitrage — Submission  |  Sunday 11 May 2026  |  Mondrian CQR bucket 1',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO / 'nb_eval_predictions.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
print('SUBMISSION SUMMARY'); print('='*50)
for zone in ZONES:
    p50  = preds[f'{zone} p50']
    p025 = preds[f'{zone} p025']
    p975 = preds[f'{zone} p975']
    print(f'\n{zone}')
    print(f'  Mean p50   : {p50.mean():.2f} EUR/MWh')
    print(f'  Min  p50   : {p50.min():.2f}  @ {preds["timestamp"][p50.idxmin()].strftime("%H:%M CEST")}')
    print(f'  Max  p50   : {p50.max():.2f}  @ {preds["timestamp"][p50.idxmax()].strftime("%H:%M CEST")}')
    print(f'  Band width : {(p975-p025).mean():.2f} EUR/MWh')
    print(f'  Negative   : {(p50 < 0).sum()} slots')